# 6 · Inferring ages

Notebook 5 produced the **PLS-3 scores** — the three latent directions that covary most
with stellar age. This notebook turns those three numbers into an **age posterior**.

The bundled model is a **neural likelihood estimator**, not a regressor. Rather than
learning `age = f(features)`, a normalizing flow learns the conditional density

$$
p(\text{PLS-3 features} \mid \log_{10}(\mathrm{age}/\mathrm{Myr}),\; \mathrm{BPRP0},\; \log_{10}\sigma_{\mathrm{BPRP0}})
$$

and an age is recovered by inverting it on a grid against a uniform prior:

$$
p(\mathrm{age} \mid \text{features}, \text{colour}) \;\propto\; p(\text{features} \mid \mathrm{age}, \text{colour}) \cdot p(\mathrm{age})
$$

Two consequences. First, the output is a **distribution**, so you get genuine uncertainty
rather than an error bar bolted on afterwards. Second, conditioning on **colour** is
deliberate: a latent's age signal is tangled up with spectral type, and supplying the
dereddened BP−RP lets the flow model age dependence *at fixed colour*.

> ### Read this before using the numbers
>
> This model was trained on **all** 9,413 labelled stars, with no held-out test set — only
> a 15% carve-out to choose the stopping epoch. It demonstrates the inference machinery;
> it is **not** a validated age predictor. The example below deliberately infers an age
> for a star the model was trained on, so the agreement you see is **in-sample and
> optimistic**. Honest accuracy needs the cross-validated pipeline, where both the PLS
> projection and the flow are refit inside every fold.
>
> The PLS projection feeding this is itself supervised (see notebook 5), which is a second
> reason not to read held-out performance into anything here.

In [ ]:
import numpy as np
import encotess
from encotess import assets

age_model = encotess.load_age_model()
print('trained on      :', age_model.n_train_stars, 'stars')
print('PLS features    :', age_model.pls_dim)
print('log-age prior   : uniform over', age_model.prior_loga_myr, 'in log10(age/Myr)')
print('                  =', [round(10**v, 1) for v in age_model.prior_loga_myr], 'Myr')

## A star to test on

Everything needed ships with the package: the per-star PLS scores, and the dereddened
colour and literature age from the metadata tables. We take the first three PLS
components — which, as notebook 5 explains, is exactly a 3-component fit of the bundled
projection.

In [ ]:
import csv

pls_file = np.load(assets.pls_encoding_path(), allow_pickle=False)
meta = {r['GaiaDR3_ID'].strip(): r
        for r in csv.DictReader(open(assets.metadata_path('FGKMcal_star'), newline=''))}

gids, feats, colour, colour_err, true_age = [], [], [], [], []
for g, p, in_fit in zip(pls_file['gaia_ids'], pls_file['pls'], pls_file['in_age_fit']):
    r = meta.get(g)
    if r is None or not in_fit:
        continue
    try:
        a, b, be = float(r['age_Myr']), float(r['BPRP0']), float(r['BPRP0_err'])
    except ValueError:
        continue
    if np.isfinite([a, b, be]).all():
        gids.append(g); feats.append(p[:3])
        colour.append(b); colour_err.append(be); true_age.append(a)

gids = np.array(gids); feats = np.array(feats)
colour = np.array(colour); colour_err = np.array(colour_err); true_age = np.array(true_age)
print(f'{len(gids)} stars with PLS scores, a colour and a literature age')

i = 0
print(f"\nexample star  Gaia DR3 {gids[i]}")
print(f"  PLS-3       {feats[i].round(3)}")
print(f"  BPRP0       {colour[i]:.3f} +/- {colour_err[i]:.4f}")
print(f"  literature  {true_age[i]:.0f} Myr  (log10 = {np.log10(true_age[i]):.3f})")

## Infer

`infer` takes the raw PLS-3 scores (it standardizes them internally, using the statistics
stored with the model) plus the colour and its error. Everything comes back in
**log10(age/Myr)**, with `age_myr` added as the median in linear units.

In [ ]:
post = age_model.infer(feats[i], colour[i], colour_err[i], return_posterior=True)

print(f"median  {post['median']:.3f}  ->  {post['age_myr']:.0f} Myr")
print(f"p16-p84 {post['p16']:.3f} - {post['p84']:.3f}"
      f"  ->  {10**post['p16']:.0f} - {10**post['p84']:.0f} Myr")
print(f"MAP     {post['map']:.3f}")
print(f"\nliterature {np.log10(true_age[i]):.3f}"
      f"   inside p16-p84: {post['p16'] <= np.log10(true_age[i]) <= post['p84']}")

The posterior is the actual product here — its width is the model telling you how much
the three PLS numbers plus a colour really pin down. Plotting it makes that concrete:

In [ ]:
import matplotlib.pyplot as plt

g, p = post['loga_grid'], post['posterior']
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(g, p, lw=1.8, label='posterior')
ax.fill_between(g, p, where=(g >= post['p16']) & (g <= post['p84']),
                alpha=0.25, label='p16-p84')
ax.axvline(post['median'], ls='--', lw=1.2, label=f"median ({post['age_myr']:.0f} Myr)")
ax.axvline(np.log10(true_age[i]), color='k', lw=1.4,
           label=f'literature ({true_age[i]:.0f} Myr)')
ax.set_xlabel('log10(age / Myr)'); ax.set_ylabel('posterior density')
ax.set_title(f'Gaia DR3 {gids[i]}')
ax.legend(fontsize=8); fig.tight_layout(); plt.show()

## A batch

`infer` vectorizes, so a whole sample runs in one call. The scatter below is **in-sample**
— every one of these stars was in the training set — so treat it as a check that the
machinery works, not as an accuracy measurement.

In [ ]:
res = age_model.infer(feats, colour, colour_err)
pred, truth = res['median'], np.log10(true_age)
resid = pred - truth

print('IN-SAMPLE (these stars were trained on)')
print(f'  n                 {len(truth)}')
print(f'  correlation       {np.corrcoef(pred, truth)[0, 1]:.3f}')
print(f'  bias              {resid.mean():+.3f} dex')
print(f'  RMSE              {np.sqrt((resid**2).mean()):.3f} dex')
print(f'  p16-p84 coverage  {((truth >= res["p16"]) & (truth <= res["p84"])).mean():.3f}'
      '   (nominal 0.68)')

In [ ]:
fig, ax = plt.subplots(figsize=(4.6, 4.6))
ax.scatter(truth, pred, s=3, alpha=0.18, edgecolors='none')
lim = [0, 4.2]
ax.plot(lim, lim, 'k--', lw=1, label='1:1')
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('literature log10(age / Myr)')
ax.set_ylabel('inferred log10(age / Myr)')
ax.set_title('In-sample — not a validation plot', fontsize=10)
ax.legend(fontsize=8); fig.tight_layout(); plt.show()

A correlation near 0.6 with ~0.68 dex scatter is what three latent directions plus a
colour buy you, *before* any held-out penalty. The coverage running above the nominal
0.68 says the posteriors are, if anything, a little conservative — wide rather than
overconfident, which is the preferable failure mode.

## Starting from your own light curve

For a star that is not in the released set, the chain is: preprocess → encode →
aggregate across sectors → project to PLS-3 → infer. The aggregation step matters —
the projection expects one row per **star**, combined with an element-wise maximum.

In [ ]:
# Sketch of the full path (uses a synthetic curve so this cell stays offline).
rng = np.random.default_rng(0)
n = 4000
t = np.arange(n) * (2.0 / 60.0 / 24.0)
f = np.sin(2 * np.pi * t / 3.0) + rng.normal(0, 1.0, n)
f = ((f - np.median(f)) / ((np.percentile(f, 84) - np.percentile(f, 16)) / 2)).astype(np.float32)
meta_dict = {'cadence_s': 120.0, 'Tmag': 10.5, 'sector': 45, 'camera': 1, 'ccd': 2,
             'parallax': 5.0, 'parallax_error': 0.02, 'G0': 10.8, 'G0_err': 0.01,
             'BPRP0': 1.1, 'BPRP0_err': 0.03, 'median_flux': 1e4, 'iqr_half_flux': 50.0}

enc = encotess.load_encoder(device='cpu')
z = enc.encode(f, np.ones(n, np.float32), t.astype(np.float32), metadata=meta_dict)

# one light curve -> one "star" here; with several sectors, aggregate first:
#   _, Z = encotess.aggregate_by_star(latents, gaia_ids, how='max')
t3 = enc.project_pls(z, dim=3)
out = age_model.infer(t3, bprp0=meta_dict['BPRP0'], bprp0_err=meta_dict['BPRP0_err'])
print('PLS-3    :', t3.round(3))
print(f"inferred : {out['age_myr']:.0f} Myr"
      f"  (p16-p84 {10**out['p16']:.0f} - {10**out['p84']:.0f} Myr)")
print('\n(synthetic curve — the number is meaningless, the plumbing is the point)')

### Related

- **[5 · The PLS projection](05_the_pls_projection.ipynb)** — where the three input
  features come from, and why the projection is supervised.
- **[1 · Preprocess a light curve](01_preprocess_a_lightcurve.ipynb)** — turning a real
  TESS download into the arrays the encoder accepts.